# LangChain: Memory

## Outline

- `ConversationBufferMemory`
- `ConversationBufferWindowMemory`
- `ConversationTokenBufferMemory`
- `ConversationSummaryBufferMemory` (summary-style memory with token limit)

> **Note:** LLMs do not always produce the same results. When you run this notebook, you may get slightly different answers than those in the video.

> This notebook uses **Anthropic Claude** (`ChatAnthropic`) like `langchain_models_prompts_output_parsers.ipynb`. The course video uses OpenAI; concepts are the same.

In [ ]:
# !pip install python-dotenv anthropic langchain langchain-anthropic tiktoken

import os
import warnings

from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())
warnings.filterwarnings("ignore")

In [ ]:
# Default model (match your other tutorial notebook)
llm_model = "claude-haiku-4-5-20251001"

## `ConversationBufferMemory`

In [ ]:
from langchain.chains import ConversationChain
from langchain.memory import ConversationBufferMemory
from langchain_anthropic import ChatAnthropic

llm = ChatAnthropic(temperature=0.0, model=llm_model, max_tokens=1024)
memory = ConversationBufferMemory()
conversation = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=True,
)

conversation.predict(input="Hi, my name is Andrew")
conversation.predict(input="What is 1+1?")
conversation.predict(input="What is my name?")

In [ ]:
print(memory.buffer)
memory.load_memory_variables({})

In [ ]:
memory = ConversationBufferMemory()
memory.save_context({"input": "Hi"}, {"output": "What's up"})
print(memory.buffer)
memory.load_memory_variables({})

In [ ]:
memory.save_context(
    {"input": "Not much, just hanging"},
    {"output": "Cool"},
)
memory.load_memory_variables({})

## `ConversationBufferWindowMemory`

In [ ]:
from langchain.memory import ConversationBufferWindowMemory

memory = ConversationBufferWindowMemory(k=1)
memory.save_context({"input": "Hi"}, {"output": "What's up"})
memory.save_context(
    {"input": "Not much, just hanging"},
    {"output": "Cool"},
)

memory.load_memory_variables({})

In [ ]:
llm = ChatAnthropic(temperature=0.0, model=llm_model, max_tokens=1024)
memory = ConversationBufferWindowMemory(k=1)
conversation = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=False,
)

conversation.predict(input="Hi, my name is Andrew")
conversation.predict(input="What is 1+1?")
conversation.predict(input="What is my name?")

## `ConversationTokenBufferMemory`

Uses **`tiktoken`** (or the LLM’s tokenizer) to cap how much of the chat is kept by **token** count.

In [ ]:
# !pip install tiktoken

from langchain.memory import ConversationTokenBufferMemory

llm = ChatAnthropic(temperature=0.0, model=llm_model, max_tokens=1024)
memory = ConversationTokenBufferMemory(llm=llm, max_token_limit=50)
memory.save_context({"input": "AI is what?!"}, {"output": "Amazing!"})
memory.save_context(
    {"input": "Backpropagation is what?"},
    {"output": "Beautiful!"},
)
memory.save_context({"input": "Chatbots are what?"}, {"output": "Charming!"})

memory.load_memory_variables({})

## `ConversationSummaryBufferMemory`

The course often labels this section “ConversationSummaryMemory”; the class used is **`ConversationSummaryBufferMemory`**, which **summarizes** older turns when approaching a token limit.

In [ ]:
from langchain.memory import ConversationSummaryBufferMemory

schedule = (
    "There is a meeting at 8am with your product team. "
    "You will need your powerpoint presentation prepared. "
    "9am-12pm have time to work on your LangChain "
    "project which will go quickly because Langchain is such a powerful tool. "
    "At Noon, lunch at the italian resturant with a customer who is driving "
    "from over an hour away to meet you to understand the latest in AI. "
    "Be sure to bring your laptop to show the latest LLM demo."
)

llm = ChatAnthropic(temperature=0.0, model=llm_model, max_tokens=1024)
memory = ConversationSummaryBufferMemory(llm=llm, max_token_limit=100)
memory.save_context({"input": "Hello"}, {"output": "What's up"})
memory.save_context(
    {"input": "Not much, just hanging"},
    {"output": "Cool"},
)
memory.save_context(
    {"input": "What is on the schedule today?"},
    {"output": f"{schedule}"},
)

memory.load_memory_variables({})

In [ ]:
conversation = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=True,
)

conversation.predict(input="What would be a good demo to show?")
memory.load_memory_variables({})

---

**Reminder:** Save this notebook locally (or commit) so your work is preserved.